[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/kgrid-objects/FAIR-DO-Workshop/HEAD?urlpath=lab/tree/collection/dfu-hbot-bounded-regimen-and-execution-burden/dfu-hbot_binder.ipynb)

## Full In-Notebook Regimen Range (Copied From src/regimen-range.js)

This section embeds the full regimen range logic directly in the notebook.


In [ ]:
const SPECIFICATION_IRI = 'https://kgrid.org/cks/dfu-hbot-burden/versions/cks-1.0';
const RESPONSE_MODEL_IRI = 'https://kgrid.org/cks/dfu-hbot-burden/response-models/1.0';
const KNOWLEDGE_PACKAGE_IRI =
  'https://kgrid.org/cks/dfu-hbot-burden/knowledge-packages/general-dfu-regimen/versions/1.0';

function showRegimenRange(request) {

  const response = {
    specification_iri: SPECIFICATION_IRI,
    response_type: 'fixed_regimen_range_response',
    response_model_iri: RESPONSE_MODEL_IRI,
    status: 'completed',
    knowledge_package_iri: KNOWLEDGE_PACKAGE_IRI,
    knowledge_scope: 'general_not_patient_specific',
    indication: 'diabetes_related_foot_ulcer',
    shorter_regimen_scenario: {
      episodes: 30,
      sessions_per_week: 5,
      course_weeks: 6.0
    },
    longer_regimen_scenario: {
      episodes: 40,
      sessions_per_week: 5,
      course_weeks: 8.0
    },
    combined_range: {
      episodes: [30, 40],
      sessions_per_week: 5,
      course_weeks: [6.0, 8.0]
    },
    scheduled_facility_hours_per_episode: {
      value: 3.0,
      unit: 'h'
    },
    tailored_to_patient: false,
    display_label:
      'General DFU HBOT planning range: approximately 30–40 episodes, usually five per week, over approximately 6–8 weeks.'
  };

  return response;
}

module.exports = {
  SPECIFICATION_IRI,
  RESPONSE_MODEL_IRI,
  KNOWLEDGE_PACKAGE_IRI,
  showRegimenRange
};


In [ ]:
const response = showRegimenRange({ request_type: 'show_regimen_range' });
console.log(JSON.stringify(response, null, 2));

## Full In-Notebook Burden Questionnaire Logic (Copied From src/questionnaire-logic.js)

This section embeds the full questionnaire logic directly in the notebook.

In [ ]:
'use strict';

const readline = require('node:readline');
const { stdin, stdout } = require('node:process');

const PROVIDER_ROSTER_VERSION_IRI =
  'https://kgrid.org/cks/dfu-hbot-burden/provider-rosters/appendix-e/versions/1.0';

const QUESTIONS = [
  {
    id: 'Q01',
    text: 'Where will you receive hyperbaric oxygen therapy?',
    prompt: 'Enter the provider IRI (for example: https://kgrid.org/cks/dfu-hbot-burden/providers/e-001):'
  },
  {
    id: 'Q02',
    text: 'About how many miles is it one way to this location?',
    prompt: 'Enter one_way_miles (0 to 1000):'
  },
  {
    id: 'Q03',
    text: 'About how long does that trip usually take one way?',
    prompt: 'Enter one_way_travel_minutes (0 to 1440):'
  },
  {
    id: 'Q04',
    text: 'How much difficulty do you expect with attending scheduled treatments?',
    prompt: 'Enter weekday_attendance_difficulty [none|some|major]:'
  }
];

function promptLine(promptText) {
  if (globalThis.$$ && typeof globalThis.$$.input === 'function') {
    return globalThis.$$.input({ prompt: promptText });
  }

  const rl = readline.createInterface({ input: stdin, output: stdout });
  return new Promise((resolve) => rl.question(promptText, (answer) => {
    rl.close();
    resolve(answer);
  }));
}

function normalizeProvider(value) {
  const providerIri = String(value || '').trim();
  if (!/^https:\/\/kgrid\.org\/cks\/dfu-hbot-burden\/providers\/e-(00[1-9]|0[1-9][0-9]|1[0-3][0-9]|14[01])$/.test(providerIri)) {
    throw new Error('Q01 must be a valid provider IRI from the Appendix E roster.');
  }
  return providerIri;
}

async function runBurdenQuestionnaire() {
  const answers = {};

  for (const question of QUESTIONS) {
    console.log(`\n${question.id}: ${question.text}`);
    const raw = String(await promptLine(`${question.prompt} `)).trim();
    if (!raw) {
      throw new Error(`${question.id} requires a response.`);
    }
    answers[question.id] = raw;
  }

  const miles = Number(answers.Q02);
  const minutes = Number(answers.Q03);
  const difficulty = answers.Q04.toLowerCase();

  if (!Number.isFinite(miles) || miles < 0 || miles > 1000) {
    throw new Error('Q02 must be a number from 0 through 1000.');
  }
  if (!Number.isFinite(minutes) || minutes < 0 || minutes > 1440) {
    throw new Error('Q03 must be a number from 0 through 1440.');
  }
  if (!['none', 'some', 'major'].includes(difficulty)) {
    throw new Error('Q04 must be none, some, or major.');
  }

  return {
    provider_roster_version_iri: PROVIDER_ROSTER_VERSION_IRI,
    response_projection: {
      hyperbaric_oxygen_therapy_location: normalizeProvider(answers.Q01),
      one_way_miles: miles,
      one_way_travel_minutes: minutes,
      weekday_attendance_difficulty: difficulty
    },
    confirmed: true
  };
}

## Simplified In-Notebook Burden Response Analysis

This demonstration keeps the core burden calculation and provider lookup while omitting schema validation, provenance, fingerprinting, and audit-only machinery.

In [ ]:
'use strict';

const providerRosterSnapshot = require('./spec/DFU_HBOT_Provider_Roster_Snapshot_Version_1_0.json');
const REPEATED_ATTENDANCE_DRIVER = 'Repeated attendance (30–40 episodes; five per week)';

const ASSUMPTIONS = [
  { code: 'ASM-01', text: 'General DFU planning regimen range of 30–40 episodes at five sessions per week is used; it was not selected or calculated for the patient.' },
  { code: 'ASM-02', text: 'A single fixed three-hour scheduled facility block per treatment episode is used as a conservative, realistic worst-case planning assumption.' },
  { code: 'ASM-03', text: 'Round trip is estimated as two times the patient-reported usual one-way miles and minutes.' },
  { code: 'ASM-04', text: 'Reported travel values are planning estimates and are not independently verified by the engine.' }
];

function endpoint(value, direction) {
  const quantum = value >= 0 && value < 10 ? 0.1 : 1;
  const rounded = direction === 'upper'
    ? Math.ceil(value / quantum) * quantum
    : Math.floor(value / quantum) * quantum;
  return Math.round(rounded * 10) / 10;
}

function nearest(value) {
  const quantum = value >= 0 && value < 10 ? 0.1 : 1;
  return Math.round((Math.floor(value / quantum + 0.5) * quantum) * 10) / 10;
}

function travelDriver(travelBand, roundTripHours) {
  const displayHours = String(nearest(roundTripHours)).replace(/\.0$/, '');
  return {
    moderate: `Moderate reported travel time (${displayHours} round-trip hours)`,
    high: `High reported travel time (${displayHours} round-trip hours)`,
    very_high: `Very-high reported travel time (${displayHours} round-trip hours)`
  }[travelBand] || null;
}

function calculateBurdenRange(questionnaireResponse, regimenRangeResponse) {
  const projection = questionnaireResponse.response_projection;
  const provider = providerRosterSnapshot.records.find(
    (record) => record.provider_iri === projection.hyperbaric_oxygen_therapy_location
  );
  if (!provider || provider.selectable !== true) {
    throw new Error('Selected provider was not found in the selectable roster.');
  }

  const roundTripHours = (projection.one_way_travel_minutes * 2) / 60;
  const roundTripMiles = projection.one_way_miles * 2;
  const scenarios = [
    { name: 'shorter_regimen', episodes: 30, weeks: 6.0, direction: 'lower' },
    { name: 'longer_regimen', episodes: 40, weeks: 8.0, direction: 'upper' }
  ];
  const burdenByScenario = {};

  for (const scenario of scenarios) {
    const travelRaw = scenario.episodes * roundTripHours;
    const facilityHours = scenario.episodes * 3;
    burdenByScenario[scenario.name] = {
      episodes: scenario.episodes,
      course_weeks: scenario.weeks,
      travel_hours: endpoint(travelRaw, scenario.direction),
      facility_hours: facilityHours,
      total_patient_hours: endpoint(travelRaw + facilityHours, scenario.direction),
      course_miles: endpoint(scenario.episodes * roundTripMiles, scenario.direction)
    };
  }

  const travelBand = roundTripHours < 1
    ? 'low'
    : roundTripHours < 2
      ? 'moderate'
      : roundTripHours < 3
        ? 'high'
        : 'very_high';
  const difficulty = projection.weekday_attendance_difficulty;
  const travelScore = { low: 1, moderate: 2, high: 3, very_high: 4 }[travelBand];
  const score = difficulty === 'major'
    ? (travelScore >= 3 ? 5 : 4)
    : difficulty === 'some' ? Math.min(5, travelScore + 1) : travelScore;
  const categories = [
    'low_execution_burden',
    'moderate_execution_burden',
    'high_execution_burden',
    'very_high_execution_burden',
    'extreme_execution_burden'
  ];
  const displayLabels = [
    'Low execution burden',
    'Moderate execution burden',
    'High execution burden',
    'Very high execution burden',
    'Extreme execution burden'
  ];

  const attendance = {
    reported_value: difficulty,
    display_text: {
      none: 'No weekday-attendance difficulty reported',
      some: 'Some weekday-attendance difficulty reported',
      major: 'Major weekday-attendance difficulty reported'
    }[difficulty]
  };
  const attendanceDriver = difficulty === 'major'
    ? 'Major weekday-attendance difficulty'
    : difficulty === 'some' ? 'Some weekday-attendance difficulty' : null;
  const travelDriverText = travelDriver(travelBand, roundTripHours);
  let primaryDriver = REPEATED_ATTENDANCE_DRIVER;
  let secondaryDriver = null;

  if (difficulty === 'major') {
    primaryDriver = attendanceDriver;
    secondaryDriver = travelScore === 1 ? REPEATED_ATTENDANCE_DRIVER : travelDriverText;
  } else if (difficulty === 'some') {
    primaryDriver = travelScore <= 2 ? attendanceDriver : travelDriverText;
    secondaryDriver = travelScore === 1 ? REPEATED_ATTENDANCE_DRIVER :
      travelScore === 2 ? travelDriverText : attendanceDriver;
  } else if (travelScore > 1) {
    primaryDriver = travelDriverText;
    secondaryDriver = REPEATED_ATTENDANCE_DRIVER;
  }

  const shorter = burdenByScenario.shorter_regimen;
  const longer = burdenByScenario.longer_regimen;
  const resultCode = `RESULT-${categories[score - 1].toUpperCase()}`;

  return {
    status: 'completed',
    result_code: resultCode,
    objective_burden: {
      fixed_regimen_range_response: regimenRangeResponse,
      hyperbaric_oxygen_therapy_location_reference: {
        provider_roster_version_iri: PROVIDER_ROSTER_VERSION_IRI,
        roster_record_iri: provider.roster_record_iri,
        provider_iri: provider.provider_iri,
        compact_id: provider.compact_id,
        facility_listing: provider.facility_listing,
        jurisdiction: provider.jurisdiction,
        address_status: provider.address_status,
        normalized_address_candidate: provider.normalized_address_candidate
      },
      reported_one_way_travel: {
        miles: projection.one_way_miles,
        minutes: projection.one_way_travel_minutes
      },
      burden_by_regimen_scenario: burdenByScenario,
      overall_burden_range: {
        episodes: [shorter.episodes, longer.episodes],
        travel_hours: [shorter.travel_hours, longer.travel_hours],
        facility_hours: [shorter.facility_hours, longer.facility_hours],
        total_patient_hours: [shorter.total_patient_hours, longer.total_patient_hours],
        course_miles: [shorter.course_miles, longer.course_miles]
      },
      travel_band: { value: travelBand, basis: 'reported_one_way_minutes_doubled' }
    },
    weekday_attendance_feasibility: attendance,
    execution_burden_level: {
      score,
      category: categories[score - 1],
      primary_driver: primaryDriver,
      secondary_driver: secondaryDriver,
      display_text: displayLabels[score - 1]
    },
    sensitivity: {
      longer_minus_shorter_episodes: 10,
      incremental_travel_hours: nearest(10 * roundTripHours),
      incremental_facility_hours: 30,
      incremental_total_hours: nearest(10 * (roundTripHours + 3)),
      incremental_miles: nearest(10 * roundTripMiles),
      incremental_weeks: 2.0
    },
    assumptions: ASSUMPTIONS,
    evidence_sources: [],
    error: null,
    warnings: []
  };
}

## Run the Questionnaire and analysis




In [ ]:
globalThis.questionnaireAndAnalysisPromise = (async () => {
  const questionnaireResponse = await runBurdenQuestionnaire();
  console.log('\nCompleted Questionnaire Response:');
  console.log(JSON.stringify(questionnaireResponse, null, 2));

  const regimenRangeResponse = showRegimenRange({ request_type: 'show_regimen_range' });
  console.log('\nFixed Regimen Range Response:');
  console.log(JSON.stringify(regimenRangeResponse, null, 2));

  const analysisResult = calculateBurdenRange(questionnaireResponse, regimenRangeResponse);
  console.log('\nBurden Analysis Result:');
  console.log(JSON.stringify(analysisResult, null, 2));

  return { questionnaireResponse, regimenRangeResponse, analysisResult };
})().catch((error) => {
  console.error(error && error.message ? error.message : error);
  throw error;
});

## Another Way in Binder: Run the CLI in a Terminal or Code Cell

In Binder, the most reliable CLI flow is still a Terminal tab:

Open a Terminal in Binder (File -> New -> Terminal).

```bash
cd collection/dfu-hbot-bounded-regimen-and-execution-burden
npm install
npx dfu-hbot-burden
```